# Tensorflow 기초와 회귀분석
강의 9장 실습 전체 재현 (Tensor, 회귀분석, Gradient Descent)

In [ ]:
# Tensorflow 설치 (주석처리)
# !pip install tensorflow pandas matplotlib

## 1. Tensor 정의와 자료형 확인

In [1]:
import tensorflow as tf

# 정수형 텐서
scalar_int = tf.constant(1)
print(scalar_int)

# 실수형 텐서
scalar_float = tf.constant(1.0, dtype=tf.float32)
print(scalar_float)

/Users/juns/business_analytics/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(1.0, shape=(), dtype=float32)


## 2. 행렬 저장 및 연산

In [2]:
A = tf.constant([[1, 2], [3, 4]])
B = tf.constant([[5, 6], [7, 8]])
print("덧셈:", tf.add(A, B))
print("곱셈:", tf.matmul(A, B))

덧셈: tf.Tensor(
[[ 6  8]
 [10 12]], shape=(2, 2), dtype=int32)
곱셈: tf.Tensor(
[[19 22]
 [43 50]], shape=(2, 2), dtype=int32)


## 3. 단위행렬과 역행렬

In [3]:
I = tf.eye(2)
print("단위행렬 곱:", tf.matmul(tf.cast(A, tf.float32), I))
A_inv = tf.linalg.inv(tf.cast(A, tf.float32))
print("역행렬 곱:", tf.matmul(tf.cast(A, tf.float32), A_inv))

단위행렬 곱: tf.Tensor(
[[1. 2.]
 [3. 4.]], shape=(2, 2), dtype=float32)
역행렬 곱: tf.Tensor(
[[ 1.0000000e+00  0.0000000e+00]
 [-4.7683716e-07  1.0000002e+00]], shape=(2, 2), dtype=float32)


## 4. 회귀분석 데이터 로드 및 시각화

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 예제용 CSV 데이터 (기온-temp / 매출-sales)
df = pd.read_csv("softdrink.csv")
x = tf.constant(df['temp'].values, dtype=tf.float32)
y = tf.constant(df['sales'].values, dtype=tf.float32)

plt.scatter(x, y)
plt.xlabel('Temperature')
plt.ylabel('Sales')
plt.title('Softdrink Sales vs Temperature')
plt.grid(True)
plt.show()

## 5. 모델 초기 설정 및 예측 함수

In [ ]:
a = tf.Variable(20.0)
b = tf.Variable(650.0)

def y_hat(x):
    return a * x + b

plt.scatter(x, y, label='True')
plt.plot(x, y_hat(x), color='red', label='Initial Model')
plt.legend()
plt.grid(True)
plt.show()

## 6. 손실함수 (MSE) 정의 및 초기 오차 측정

In [ ]:
def loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

print("초기 MSE:", loss(y, y_hat(x)).numpy())

## 7. GradientTape로 그래디언트 계산

In [ ]:
with tf.GradientTape() as tape:
    loss_val = loss(y, y_hat(x))
da, db = tape.gradient(loss_val, [a, b])
print("da:", da.numpy(), "db:", db.numpy())

## 8. 경사하강법 최적화 구현

In [ ]:
learning_rate = 0.001
loss_history = []

for step in range(10000):
    with tf.GradientTape() as tape:
        loss_val = loss(y, y_hat(x))
    da, db = tape.gradient(loss_val, [a, b])
    a.assign_sub(learning_rate * da)
    b.assign_sub(learning_rate * db)
    loss_history.append(loss_val.numpy())

    if tf.abs(da) < 0.05 and tf.abs(db) < 0.05:
        break

print(f"최종 a = {a.numpy():.4f}, b = {b.numpy():.4f}, 최종 MSE = {loss(y, y_hat(x)).numpy():.4f}")

## 9. 최종 예측 결과 시각화

In [ ]:
plt.scatter(x, y, label='True')
plt.plot(x, y_hat(x), color='green', label='Final Model')
plt.legend()
plt.grid(True)
plt.title("Final Regression Fit")
plt.show()

## 10. 손실함수 변화 시각화

In [ ]:
plt.plot(loss_history)
plt.title("Loss over Iterations")
plt.xlabel("Step")
plt.ylabel("MSE")
plt.grid(True)
plt.show()